# ML-09 — Validation and Research Claim Audit

The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.

## 1. Two paper findings + my methodology questions

### Finding 1 — Content age and decline

FlyRank reports that declining pages were older on average than growing pages.
The report measured average age of about 185 days for growing pages and about
228 days for declining pages.

My methodology question is: does this relationship remain when pages are
compared across different clients rather than treating all pages as independent?
Client-level differences in content strategy, traffic and publishing schedules
could affect the observed relationship. I would therefore treat this finding as
an observed association rather than proof that age itself causes decline.

### Finding 2 — CTR and ranking position

FlyRank reports that click-through rate changes substantially across ranking
position tiers, with much stronger click capture near the top of search results
than on deeper pages.

My methodology question is: are the position and CTR measurements calculated
using the same performance window and are they weighted consistently across
pages? A page's search volume and impression count can strongly affect aggregate
CTR, so the validation should make sure that the comparison is not dominated by
a small number of high-volume pages.

These questions are constructive checks on methodology rather than claims that
the findings are wrong.

In [1]:
# Load the same CSV used throughout the project

import pandas as pd
import numpy as np

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFinding 1 variables available:")
print("content_age_days:", "content_age_days" in df.columns)
print("trend_direction:", "trend_direction" in df.columns)

print("\nFinding 2 variables available:")
print("ctr:", "ctr" in df.columns)
print("avg_position:", "avg_position" in df.columns)

# Show the relevant fields if they exist

finding_columns = [
    col for col in [
        "content_id",
        "client_id",
        "content_age_days",
        "trend_direction",
        "ctr",
        "avg_position",
        "impressions_90d",
        "clicks_90d"
    ]
    if col in df.columns
]

display(df[finding_columns].head(10))

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Finding 1 variables available:
content_age_days: True
trend_direction: True

Finding 2 variables available:
ctr: True
avg_position: True


,content_id,client_id,content_age_days,trend_direction,ctr,avg_position,impressions_90d,clicks_90d
0,content_304f48230142,client_f369cb89fc,187,down,0.76,10.6,3803,29
1,content_a1fb4e703a9e,client_4e07408562,445,down,0.05,20.3,15320,7
2,content_9aa793d4d895,client_7f2253d7e2,141,down,0.09,36.5,12581,11
3,content_331d6c4de07b,client_19581e27de,463,stable,0.49,6.2,11751,58
4,content_d99b7a2d90ca,client_3fdba35f04,263,down,0.13,44.0,19140,24
5,content_d4084a4bc775,client_f369cb89fc,147,down,0.03,8.5,3970,1
6,content_9a34b442b552,client_8722616204,90,down,0.00,7.0,20,0
7,content_a63219c6e95a,client_19581e27de,445,stable,0.06,21.2,1724,1
8,content_5e6c160719bc,client_6208ef0f77,90,down,0.09,46.0,32574,29
9,content_c27558df2b0c,client_19581e27de,257,down,0.16,4.9,1240,2


## 2. My model under an honest split (before/after)

### Honest validation design

The CSV does not contain a usable event-date column for a time-aware split.
Instead, I use a grouped split by `client_id`.

This prevents pages from the same client appearing in both the training and
test sets. The purpose is to test whether the model generalizes to unseen
clients rather than simply memorizing client-specific patterns.

The original Week-5 model used a regular train/test split. Here I compare that
result with the grouped-client split. The grouped result is the more honest
estimate for cross-client generalization.

In [2]:
# Honest grouped-client validation

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Reload data so this cell can run independently
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# Create the target

# Declining content = positive class
# We do NOT use trend_direction as an input feature.
if "trend_direction" not in df.columns:
    raise ValueError("trend_direction column is required for the target.")

df["target"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Select leakage-safe features

candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

feature_columns = [
    col for col in candidate_features
    if col in df.columns
]

X = df[feature_columns].copy()

# Keep only numeric features
X = X.select_dtypes(include=[np.number])

# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values
X = X.fillna(X.median())

y = df["target"]

print("Features used:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

# BEFORE: regular random split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler_before = StandardScaler()

X_train_scaled = scaler_before.fit_transform(X_train)
X_test_scaled = scaler_before.transform(X_test)

model_before = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_before.fit(X_train_scaled, y_train)

pred_before = model_before.predict(X_test_scaled)

before_f1 = f1_score(
    y_test,
    pred_before,
    zero_division=0
)

before_accuracy = accuracy_score(
    y_test,
    pred_before
)

# AFTER: grouped split by client

if "client_id" not in df.columns:
    raise ValueError("client_id column is required for grouped validation.")

groups = df["client_id"].astype(str)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

scaler_after = StandardScaler()

X_train_group_scaled = scaler_after.fit_transform(
    X_train_group
)

X_test_group_scaled = scaler_after.transform(
    X_test_group
)

model_after = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_after.fit(
    X_train_group_scaled,
    y_train_group
)

pred_after = model_after.predict(
    X_test_group_scaled
)

after_f1 = f1_score(
    y_test_group,
    pred_after,
    zero_division=0
)

after_accuracy = accuracy_score(
    y_test_group,
    pred_after
)

# Check client overlap

train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

overlap = train_clients.intersection(test_clients)

print("\nGrouped validation")
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(overlap))

# Comparison table

comparison = pd.DataFrame({
    "Split": [
        "Random 80/20",
        "Grouped by client"
    ],
    "Accuracy": [
        before_accuracy,
        after_accuracy
    ],
    "F1": [
        before_f1,
        after_f1
    ]
})

display(comparison)

print("\nThe grouped split is the more conservative estimate because")
print("the model is evaluated on clients not seen during training.")

Features used:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Target distribution:
target
1    16262
0    13738
Name: count, dtype: int64

Grouped validation
Training clients: 25
Testing clients: 7
Client overlap: 0


,Split,Accuracy,F1
0,Random 80/20,0.638167,0.681251
1,Grouped by client,0.578452,0.585249



The grouped split is the more conservative estimate because
the model is evaluated on clients not seen during training.


### Leakage audit

I checked the final feature set for columns that directly describe the outcome
or were constructed from the same trend information used to define the target.

`trend_direction` is excluded because it directly defines the target. 
`trend_pct` is also excluded because it describes the same recent performance
change and could reveal the outcome.

The tier and action/flag columns are also excluded where they represent
downstream decisions or labels rather than information available independently
at prediction time.

The remaining features describe search visibility, content characteristics,
engagement and freshness signals available before making the decision.

A grouped client split is also used to reduce leakage caused by having related
pages from the same client in both training and testing.

In [3]:
# Leakage audit

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# Columns that should NOT be used as model inputs
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

present_leakage_columns = [
    col for col in leakage_candidates
    if col in df.columns
]

print("Potential leakage / downstream columns found:")
print(present_leakage_columns)

# Final feature list

safe_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

safe_features = [
    col for col in safe_features
    if col in df.columns
]

# Make sure no known leakage column slipped into the final list
leaked_features_in_final_set = [
    col for col in safe_features
    if col in present_leakage_columns
]

print("\nFinal feature count:", len(safe_features))
print("Final features:")
print(safe_features)

print("\nLeakage columns in final feature set:")
print(leaked_features_in_final_set)

if len(leaked_features_in_final_set) == 0:
    print("\nLEAKAGE AUDIT: PASSED")
else:
    print("\nLEAKAGE AUDIT: FAILED")

Potential leakage / downstream columns found:
['trend_direction', 'trend_pct', 'age_tier', 'age_tier_order', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Final feature count: 21
Final features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Leakage columns in final feature set:
[]

LEAKAGE AUDIT: PASSED


## 4. Claim rewrite
### Claim rewrite

Original claim:

> The model predicts which content will decline and identifies the pages that
> should be refreshed.

Safer claim:

> The model showed a measured ability to distinguish declining from non-declining
> content in this dataset. Under grouped client validation, the result provides
> directional decision-support for prioritizing pages for human review, rather
> than proving that the model will predict future content performance or that a
> refresh will cause improvement.

### Interpretation

The grouped validation is important because the model is evaluated on clients
that were not used during training. Any reduction in performance compared with
the random split is treated as evidence that the random split was more
optimistic, not as a reason to prefer the larger number.

The model should therefore be used as decision-support for prioritization and
human review, not as an automatic content-refresh decision.
*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# Final claim-support summary
print("FINAL CLAIM AUDIT")
print("=" * 60)

print(
    "The model was evaluated using both a random split and a "
    "grouped-by-client split."
)

print(
    f"Random-split F1: {before_f1:.4f}"
)

print(
    f"Grouped-client F1: {after_f1:.4f}"
)

print(
    f"Random-split accuracy: {before_accuracy:.4f}"
)

print(
    f"Grouped-client accuracy: {after_accuracy:.4f}"
)

print("\nLeakage audit:")
if len(leaked_features_in_final_set) == 0:
    print("PASSED — no identified outcome/downstream columns are in the final feature set.")
else:
    print("FAILED — leakage columns remain:", leaked_features_in_final_set)

print("\nFinal interpretation:")
print(
    "The model provides directional decision-support for prioritizing "
    "content for human review. The results do not establish causation "
    "or guarantee future content performance."
)

FINAL CLAIM AUDIT
The model was evaluated using both a random split and a grouped-by-client split.
Random-split F1: 0.6813
Grouped-client F1: 0.5852
Random-split accuracy: 0.6382
Grouped-client accuracy: 0.5785

Leakage audit:
PASSED — no identified outcome/downstream columns are in the final feature set.

Final interpretation:
The model provides directional decision-support for prioritizing content for human review. The results do not establish causation or guarantee future content performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.